# 06 Deep Learning — Data Loading & Text Preprocessing

⚠️ **IMPORTANT: DO NOT RE-RUN THIS NOTEBOOK ACCIDENTALLY**

This notebook downloads ~100 k conversations from Hugging Face and saves
preprocessed tensors to `results/`.  Only re-run if you need fresh data.

---

## Purpose

Prepare the raw SaaS sales conversation text for two deep learning models:

1. **Custom word-level tokenizer** → padded integer tensors (for BiLSTM)
2. **HuggingFace WordPiece tokenizer** → input_ids + attention_mask (for DistilBERT)

The same **stratified 80/20 split** (`random_state=42`) from Phase 1 is used so that
all models are evaluated on an identical test set.

### Outputs saved to `results/`

| File | Contents |
|------|----------|
| `dl_train_ids.pt` / `dl_val_ids.pt` / `dl_test_ids.pt` | Padded word-level token tensors |
| `dl_y_train.npy` / `dl_y_val.npy` / `dl_y_test.npy` | Label arrays |
| `dl_bert_train.pt` / `dl_bert_val.pt` / `dl_bert_test.pt` | DistilBERT tokenizer outputs |
| `dl_tokenizer.json` | Custom tokenizer vocabulary |

## 1. Imports and Setup

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import run_data_preparation
from src.deep_learning import TokenizerConfig, SalesTokenizer

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 5)

RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
for d in [RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(exist_ok=True)

print(f'PyTorch {torch.__version__}')
print(f'Project root: {PROJECT_ROOT}')

ModuleNotFoundError: No module named 'torch'

## 2. Load Dataset from Hugging Face

We load the **same dataset** (`DeepMostInnovations/saas-sales-conversations`) used
throughout Phase 1.  The cleaning pipeline in `src/data_preparation.py` handles
missing values, duplicates, and type coercion.

In [ ]:
print('Loading dataset from Hugging Face...')
df = run_data_preparation()

TEXT_COL = 'full_text' if 'full_text' in df.columns else 'conversation_text'
TARGET_COL = 'outcome'

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
df = df.dropna(subset=[TEXT_COL, TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

texts = df[TEXT_COL].fillna('').astype(str).values
labels = df[TARGET_COL].values

print(f'\nDataset: {len(texts):,} samples')
print(f'Text column: {TEXT_COL}')
print(f'Class distribution: Lost={np.sum(labels==0):,}  Won={np.sum(labels==1):,}')

## 3. Train / Val / Test Split

| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 72 % | Model training |
| **Val** | 8 % | Early stopping & LR scheduling |
| **Test** | 20 % | Final evaluation (identical to Phase 1) |

The **first split** (`test_size=0.2, random_state=42, stratify`) matches Phase 1
notebooks 04/05 exactly, so the test set is identical.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels,
)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_text, y_train, test_size=0.1, random_state=42, stratify=y_train,
)

print(f'Train: {len(X_train_text):,}  |  Val: {len(X_val_text):,}  |  Test: {len(X_test_text):,}')
print(f'\nClass balance (train): Lost={np.sum(y_train==0):,}  Won={np.sum(y_train==1):,}')

## 4. Text Statistics

Understanding text length distribution is critical for choosing `max_length`
(truncation) and anticipating where models may struggle on very short or very
long conversations.

In [ ]:
train_word_counts = np.array([len(t.split()) for t in X_train_text])

print('Word-count statistics (training set):')
print(f'  Min:    {train_word_counts.min():>8,}')
print(f'  Mean:   {train_word_counts.mean():>8,.0f}')
print(f'  Median: {np.median(train_word_counts):>8,.0f}')
print(f'  95th %%: {np.percentile(train_word_counts, 95):>8,.0f}')
print(f'  Max:    {train_word_counts.max():>8,}')
print(f'  Texts > 512 words: {(train_word_counts > 512).sum():,} '
      f'({(train_word_counts > 512).mean():.1%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_word_counts, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(512, color='red', linestyle='--', label='max_length=512')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Conversation Lengths')
axes[0].legend()

axes[1].hist(train_word_counts[train_word_counts <= 1000], bins=80,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(512, color='red', linestyle='--', label='max_length=512')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Zoomed (≤ 1000 words)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_text_length_distribution.png')

## 5. Custom Word-Level Tokenizer (for BiLSTM)

### Design choices

| Choice | Rationale |
|--------|-----------|
| **Word-level** (not sub-word) | SaaS jargon like *"upsell"*, *"ARR"*, *"churn"* is preserved as single tokens; sub-word tokenizers would fragment them |
| **`min_freq=2`** | Removes typos and singleton tokens that add noise |
| **`max_vocab=30,000`** | Covers >99 % of running tokens in a business-English corpus while keeping the embedding matrix practical |
| **`max_length=512`** | Truncates very long conversations; empirically, opening and closing language is the most predictive |
| **Fit on training set only** | Prevents data leakage from validation/test vocabularies |

In [ ]:
tok_config = TokenizerConfig(max_vocab=30_000, max_length=512, min_freq=2)
tokenizer = SalesTokenizer(tok_config)
tokenizer.fit(X_train_text.tolist())

print(f'Vocabulary size: {tokenizer.vocab_size:,}')
print(f'Special tokens: PAD=0, UNK=1')
print(f'Max sequence length: {tok_config.max_length}')

# Show sample tokenization
sample = X_train_text[0][:200]
sample_ids = tokenizer.encode(sample)
print(f'\nSample text (first 200 chars):\n  {sample}')
print(f'\nToken IDs (first 20): {sample_ids[:20]}')

In [ ]:
# Tokenize and pad all three splits
print('Tokenizing train split...')
train_ids = tokenizer.encode_batch(X_train_text.tolist())
print('Tokenizing val split...')
val_ids = tokenizer.encode_batch(X_val_text.tolist())
print('Tokenizing test split...')
test_ids = tokenizer.encode_batch(X_test_text.tolist())

print(f'\nTensor shapes:')
print(f'  Train: {train_ids.shape}')
print(f'  Val:   {val_ids.shape}')
print(f'  Test:  {test_ids.shape}')

# Effective sequence length stats (non-padding)
train_seq_lens = (train_ids != 0).sum(dim=1).float()
print(f'\nEffective sequence lengths (after tokenization):')
print(f'  Mean: {train_seq_lens.mean():.0f}  Median: {train_seq_lens.median():.0f}  '
      f'Max: {train_seq_lens.max():.0f}')